# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library, referencing all dataset entities by their `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```python
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")
print(f"Dataset published: {metadata['datePublished']}")
print(f"Dataset license: {metadata['license']}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

The `mlcroissant` API lets us inspect record sets and their fields programmatically using their `@id` values.

In [ ]:
# List all available record sets and their fields by @id
record_sets = dataset.record_sets

print("Available Record Sets:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | name: {rs.get('name', '')}")

# For demonstration, print each field @id and name in each record set
for rs in record_sets:
    print(f"\nFields in Record Set {rs['@id']}:")
    fields = rs.get('fields', [])
    for field in fields:
        print(f"  - Field @id: {field['@id']} | name: {field.get('name', '')}")

## Example Record Access
View a few records for one record set using its `@id`. Replace `<record_set_id>` with your desired record set ID from above.

In [ ]:
# Choose the main record set for clinical and molecular analysis
clinical_rs_id = None
for rs in dataset.record_sets:
    # Use the most relevant record set for oncology and molecular variables
    if 'colorectal' in rs.get('name', '').lower() or 'clinicopathological' in rs.get('name', '').lower():
        clinical_rs_id = rs['@id']
        break
if not clinical_rs_id:
    # Fallback: just pick the first one if none matches
    clinical_rs_id = dataset.record_sets[0]['@id']

print(f"Showing records from Record Set @id: {clinical_rs_id}")

# Display some records as examples, referencing only by @id
for idx, record in enumerate(dataset.records(record_set=clinical_rs_id)):
    print(record)
    if idx >= 2:
        break

## 3. Data Extraction
Load data from the identified record sets into Pandas DataFrames, referencing each by `@id`.

In [ ]:
# Collect all record set @ids to load their data
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show columns of the main clinical record set
print("Columns in main clinical DataFrame:")
print(dataframes[clinical_rs_id].columns.tolist())
dataframes[clinical_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping.

**Refer to field columns by `@id` only!**

In [ ]:
# Find a numeric field for filtering: Try to find 'age', 'interval_years', or similar by @id
numeric_field_id = None
group_field_id = None
fields = None
for rs in dataset.record_sets:
    if rs['@id'] == clinical_rs_id:
        fields = rs.get('fields', [])
        break

# Look for a numeric field
for field in fields:
    if field.get('dataType', '').lower() in ['integer', 'float']:
        if 'interval' in field.get('name', '').lower() or 'age' in field.get('name', '').lower():
            numeric_field_id = field['@id']
            break
if not numeric_field_id:
    # Fallback: pick the first numeric field
    for field in fields:
        if field.get('dataType', '').lower() in ['integer', 'float']:
            numeric_field_id = field['@id']
            break

# Look for group field: e.g., sex or anatomical location
for field in fields:
    if 'sex' in field.get('name', '').lower():
        group_field_id = field['@id']
        break
if not group_field_id:
    for field in fields:
        if 'anatomical' in field.get('name', '').lower():
            group_field_id = field['@id']
            break

print(f"Using numeric field @id: {numeric_field_id}")
print(f"Using group field @id: {group_field_id}")

# Filtering and normalization
threshold = 10
df = dataframes[clinical_rs_id]
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Grouping
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())
else:
    print(f"Numeric field @id {numeric_field_id} not found in columns. Columns found: {df.columns.tolist()}")

## 5. Visualization
Visualize relationships between fields, referencing by `@id` as required.

In [ ]:
# Visualize distribution of the numeric field
if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    df[numeric_field_id].hist(bins=12)
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

# Visualize mean numeric field by group
if group_field_id and group_field_id in df.columns:
    grouped = df.groupby(group_field_id)[numeric_field_id].mean()
    grouped.plot(kind='bar', figsize=(8, 5))
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we loaded, explored, and processed the FAIR^2 dataset package using `mlcroissant`, referencing all entities strictly by their `@id`. We filtered and normalized a numeric field, grouped and visualized key relationships, and observed how to structure downstream analysis with transparent, schema-driven referencing.

The workflow demonstrated:
- Programmatic schema-driven data access using Croissant identifiers
- Basic EDA, filtering, normalization, and grouping
- Data visualization for clinical oncology

For more advanced analysis, extended statistical processing, and domain-specific exploration, continue referencing fields, columns, and record sets by their `@id` as defined in the Croissant schema.